# NLP Task 4: Fine-Tuning BERT for Sentiment Analysis

In this project, we fine-tune a pre-trained BERT model on a real-world dataset to perform text classification.

Pipeline:
Raw Data → Preprocessing → Tokenization → Model Training → Evaluation → Comparison

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/IMDB Dataset.csv",
    engine='python',
    encoding='latin-1',
    on_bad_lines='skip'
)



In [ ]:
df.head()

## Data Preprocessing

In this step, we clean the dataset by:
- Removing missing values
- Converting sentiment labels into numerical format

In [ ]:
df.isnull().sum()

In [ ]:
# Remove missing values (if any)
df = df.dropna()

# Convert sentiment to numeric
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

# Check result
df.head()

In [ ]:
df['sentiment'].value_counts()

## Data Splitting

In this step, we split the dataset into training, validation, and test sets to properly evaluate model performance.

In [ ]:
from sklearn.model_selection import train_test_split

# First split: Train (70%) and Temp (30%)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['sentiment'],
    test_size=0.3,
    random_state=42
)

# Second split: Validation (15%) and Test (15%)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.5,
    random_state=42
)

In [ ]:
print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

## Tokenization using BERT

In this step, we convert text data into token IDs using the BERT tokenizer.
This prepares the data for input into the BERT model.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
len(train_encodings['input_ids'])

## Creating Dataset Class

In this step, we convert tokenized data into a PyTorch Dataset format, which is required for training the BERT model.

In [ ]:
import torch
from torch.utils.data import Dataset

class IMDbDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {}
        for key in self.encodings:
            item[key] = torch.tensor(self.encodings[key][idx])

        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [ ]:
train_dataset[0]

## Model Building (BERT)

In this step, we load a pre-trained BERT model and modify it for sentiment classification.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
print(model)

## Model Training (Fine-Tuning BERT)

In this step, we fine-tune the pre-trained BERT model using the training dataset.
We use AdamW optimizer and train for a few epochs.

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
val_dataset = torch.utils.data.Subset(val_dataset, range(1000))
val_loader = DataLoader(val_dataset, batch_size=16)

## Model Evaluation

In this step, we evaluate the performance of the fine-tuned BERT model using standard classification metrics.

In [ ]:
from torch.utils.data import Subset, DataLoader
small_val_dataset = Subset(val_dataset, range(1000))

val_loader = DataLoader(small_val_dataset, batch_size=32)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    for batch in val_loader:

        inputs = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**inputs)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(inputs['labels'].cpu().numpy())

In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_labels, predictions)
print("Confusion Matrix:\n", cm)

## Insights and Analysis

- The BERT model was fine-tuned on the IMDB movie review dataset for sentiment classification.  
- The model achieved **Accuracy: 0.475**, **Precision: 0.556**, **Recall: 0.047**, and **F1 Score: 0.087**, showing that it struggled to correctly classify sentiments.  
- Confusion matrix indicates **high misclassification**, suggesting the need for more epochs, larger batch sizes, or layer-specific fine-tuning.  
- Potential improvements:
  - Train for more epochs and tune learning rate.  
  - Freeze lower BERT layers and fine-tune only the top layers.  
  - Experiment with **DistilBERT** or **RoBERTa** for faster and potentially better performance.  
- Overall, the project demonstrates a **complete BERT fine-tuning pipeline** from preprocessing to evaluation, fulfilling assignment objectives.